# Introduction
This notebook is used to fetch, plot, and analyze experiment results.

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [3]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Fetch Results

In [5]:
import seml
import pandas as pd

db_collection = 'llama-typo-eval'
#states=["FAILED"]
states = ["COMPLETED"]

all_results_llama = seml.evaluation.get_results(db_collection, to_data_frame=True, states=states)
print(f"Lenght of all_results: {len(all_results_llama)}")
print(all_results_llama.columns)

all_results_llama.head()

Output()

Output()

Lenght of all_results: 62
Index(['_id', 'config.overwrite', 'config.db_collection',
       'config.dataset_name', 'config.device', 'config.exp_id',
       'config.max_entries', 'config.max_new_tokens', 'config.model_name',
       'config.n_beams', 'config.n_repeats', 'config.num_excel_rows',
       'config.save_excel', 'config.seed', 'config.strategy',
       'config.temperature', 'config.typo_intensity', 'config.typo_type',
       'config.use_beam_search', 'result.AUCROC_sample', 'result.AUCPR_sample',
       'result.Brier_sample', 'result.LogLoss_sample', 'result.Entropy_sample',
       'result.AUCROC_adj', 'result.AUCPR_adj', 'result.Brier_adj',
       'result.LogLoss_adj', 'result.Entropy_adj', 'result.AUCROC_sem',
       'result.AUCPR_sem', 'result.Brier_sem', 'result.LogLoss_sem',
       'result.Entropy_sem', 'result.Accuracy', 'result.fail_trace'],
      dtype='object')


,_id,config.overwrite,config.db_collection,config.dataset_name,config.device,config.exp_id,config.max_entries,config.max_new_tokens,config.model_name,config.n_beams,...,result.Brier_adj,result.LogLoss_adj,result.Entropy_adj,result.AUCROC_sem,result.AUCPR_sem,result.Brier_sem,result.LogLoss_sem,result.Entropy_sem,result.Accuracy,result.fail_trace
0,1,1,llama-typo-eval,P17,cuda,typo-test-10-01,None,25,Llama-3-8B,5,...,0.246302,3.540726,293.098963,0.997062,0.998746,0.019258,0.056364,371.496660,0.737527,<function get_results at 0x7fb4b81581f0>
1,2,2,llama-typo-eval,P17,cuda,typo-test-10-01,None,25,Llama-3-8B,5,...,0.246423,4.061702,278.056702,0.997286,0.998845,0.018462,0.053767,356.393999,0.738387,<function get_results at 0x7fb4b81581f0>
2,3,3,llama-typo-eval,P17,cuda,typo-test-10-01,None,25,Llama-3-8B,5,...,0.246302,3.540726,293.098963,0.997062,0.998746,0.019258,0.056364,371.496660,0.737527,<function get_results at 0x7fb4b81581f0>
3,4,4,llama-typo-eval,P17,cuda,typo-test-10-01,None,25,Llama-3-8B,5,...,0.303752,4.207371,323.975687,0.997618,0.998587,0.018355,0.053010,355.801050,0.673226,<function get_results at 0x7fb4b81581f0>
4,5,5,llama-typo-eval,P17,cuda,typo-test-10-01,None,25,Llama-3-8B,5,...,0.363950,5.306763,348.810634,0.996775,0.997559,0.022312,0.064880,428.063428,0.611075,<function get_results at 0x7fb4b81581f0>


In [ ]:
import seml
import pandas as pd

db_collection = 'llama-quant-eval'
#states=["FAILED"]
states = ["COMPLETED"]

all_results_awq = seml.evaluation.get_results(db_collection, to_data_frame=True, states=states)
print(f"Lenght of all_results: {len(all_results_awq)}")
print(all_results_awq.columns)

all_results_awq.head()

## 3. Plot results

## 3.1 Final plots

In [8]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from fpdf import FPDF
import numpy as np

# Define custom colors
color_llama = '#7FFFD4'  # Aquamarine
color_awq = '#FF69B4'  # Hot Pink

def create_comparison_radar_plots(df_llama, df_awq, metric):
    fig = make_subplots(rows=1, cols=3, specs=[[{'type': 'polar'}]*3],
                        subplot_titles=[f"Intensity {i}" for i in [1, 2, 3]])
    
    for i, intensity in enumerate([1, 2, 3], 1):
        df_llama_intensity = df_llama[df_llama['config.typo_intensity'] == intensity]
        df_awq_intensity = df_awq[df_awq['config.typo_intensity'] == intensity]
        
        values_llama = df_llama_intensity[f'result.{metric}'].tolist()
        values_awq = df_awq_intensity[f'result.{metric}'].tolist()
        
        # Add an extra point to close the polygon
        values_llama.append(values_llama[0])
        values_awq.append(values_awq[0])
        
        theta = df_llama_intensity['config.typo_type'].tolist() + [df_llama_intensity['config.typo_type'].iloc[0]]
        
        fig.add_trace(go.Scatterpolar(
            r=values_llama,
            theta=theta,
            fill='toself',
            name='Llama-3-8B',
            line=dict(color=color_llama),
            showlegend=(i == 1)
        ), row=1, col=i)
        
        fig.add_trace(go.Scatterpolar(
            r=values_awq,
            theta=theta,
            fill='toself',
            name='AWQ-quantized',
            line=dict(color=color_awq),
            showlegend=(i == 1)
        ), row=1, col=i)
    
    fig.update_layout(
        height=500,
        width=1500,
        title=f'Comparison of {metric} for Llama-3-8B and AWQ-quantized',
    )
    return fig

def create_boxplot_comparison(df_llama, df_awq, metric):
    fig = go.Figure()
    
    for model, df, color in [('Llama-3-8B', df_llama, color_llama), ('AWQ-quantized', df_awq, color_awq)]:
        for intensity in [1, 2, 3]:
            df_intensity = df[df['config.typo_intensity'] == intensity]
            fig.add_trace(go.Box(
                y=df_intensity[f'result.{metric}'],
                x=df_intensity['config.typo_intensity'],
                name=f'{model} (Intensity {intensity})',
                marker_color=color,
                showlegend=intensity == 1
            ))
    
    fig.update_layout(
        title=f'Distribution of {metric} across Intensities',
        xaxis_title='Intensity',
        yaxis_title=metric,
        boxmode='group',
        height=600,
        width=1000
    )
    return fig

def create_heatmap_comparison(df_llama, df_awq, metric):
    fig = make_subplots(rows=1, cols=2, subplot_titles=['Llama-3-8B', 'AWQ-quantized'])
    
    for i, (df, title) in enumerate([(df_llama, 'Llama-3-8B'), (df_awq, 'AWQ-quantized')], 1):
        pivot = df.pivot(index='config.typo_type', columns='config.typo_intensity', values=f'result.{metric}')
        
        heatmap = go.Heatmap(
            z=pivot.values,
            x=pivot.columns,
            y=pivot.index,
            colorscale='Viridis',
            showscale=(i == 2),
        )
        
        fig.add_trace(heatmap, row=1, col=i)
    
    fig.update_layout(
        height=800,
        width=1500,
        title=f'Heatmap of {metric} for Different Perturbation Types and Intensities'
    )
    return fig

def generate_pdf_report(plots, pdf_path, model_names):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    # Add title page
    pdf.add_page()
    pdf.set_font("Arial", 'B', size=16)
    pdf.cell(0, 10, "Model Comparison: Typo Effect Analysis", ln=True, align='C')
    pdf.set_font("Arial", size=12)
    pdf.cell(0, 10, f"Models compared: {' vs. '.join(model_names)}", ln=True, align='C')

    # Add plots to the PDF
    for plot_file, desc in plots:
        pdf.add_page()
        pdf.set_font("Arial", 'B', size=14)
        pdf.multi_cell(0, 10, desc)
        pdf.ln(5)
        pdf.image(plot_file, w=pdf.w - 20)

    pdf.output(pdf_path, "F")

def main(all_results_llama, all_results_awq, exp_id):
    plots_dir = f"plots/model_comparison_{exp_id}"
    os.makedirs(plots_dir, exist_ok=True)

    plots = []
    metrics = ['Accuracy', 'AUCPR_sample', 'AUCPR_adj', 'LogLoss_sample', 'LogLoss_adj']

    for metric in metrics:
        # Comparison Radar Plots
        fig = create_comparison_radar_plots(all_results_llama, all_results_awq, metric)
        plot_file = os.path.join(plots_dir, f"{metric}_Comparison_Radar_Plots_{exp_id}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f"Comparison Radar Plots of {metric}"))
        
        # Boxplot Comparison
        fig = create_boxplot_comparison(all_results_llama, all_results_awq, metric)
        plot_file = os.path.join(plots_dir, f"{metric}_Boxplot_Comparison_{exp_id}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f"Boxplot Comparison of {metric}"))
        
        # Heatmap Comparison
        fig = create_heatmap_comparison(all_results_llama, all_results_awq, metric)
        plot_file = os.path.join(plots_dir, f"{metric}_Heatmap_Comparison_{exp_id}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f"Heatmap Comparison of {metric}"))

    # Generate the PDF report
    pdf_path = os.path.join(plots_dir, f"model_comparison_report_{exp_id}.pdf")
    generate_pdf_report(plots, pdf_path, ['Llama-3-8B', 'AWQ-quantized Llama-3-8B'])

    print(f"PDF report generated and saved as '{pdf_path}'")

if __name__ == "__main__":
    # Load your results dataframes here
    # Assuming you've already run the seml code to get all_results_llama and all_results_awq
    # all_results_llama = ... (your existing code to load the Llama-3-8B results)
    # all_results_awq = ... (your existing code to load the AWQ-quantized results)
    
    # Specify the experiment ID
    exp_id = "llama_vs_awq_comparison"  # You can change this to any string you want
    
    # Call the main function with the experiment ID
    main(all_results_llama, all_results_awq, exp_id)

PDF report generated and saved as 'plots/typo_effect_analysis_awq-pert-10-03/typo_effect_analysis_report_awq-pert-10-03.pdf'


## 4. Remove Duplicates

In [4]:
import pandas as pd

def inspect_and_remove_duplicates(df):
    # Define the columns used for pivoting
    pivot_columns = ['config.typo_type', 'config.typo_intensity']
    
    # Find duplicates in pivot columns
    duplicate_mask = df.duplicated(subset=pivot_columns, keep=False)
    duplicates = df[duplicate_mask]
    
    if duplicates.empty:
        print("No duplicates found in pivot columns.")
        return df
    
    print("Duplicate entries found in pivot columns:")
    print(duplicates[pivot_columns])
    
    print("\nFull rows for duplicate entries:")
    print(duplicates)
    
    # Ask user how to handle duplicates
    print("\nHow would you like to handle these duplicates?")
    print("1: Keep first occurrence")
    print("2: Keep last occurrence")
    print("3: Remove all duplicates")
    print("4: Do nothing (keep all)")
    
    choice = input("Enter your choice (1-4): ")
    
    if choice == '1':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep='first')
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '2':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep='last')
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '3':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep=False)
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '4':
        df_cleaned = df
        print("No rows removed.")
    else:
        print("Invalid choice. No rows removed.")
        df_cleaned = df
    
    return df_cleaned

# Assuming your dataframe is named 'all_results'
all_results_cleaned = inspect_and_remove_duplicates(all_results)

# You can now use all_results_cleaned for further processing

Duplicate entries found in pivot columns:
           config.typo_type  config.typo_intensity
42  word_phrase_translation                      1
43  word_phrase_translation                      2
59  word_phrase_translation                      1
60  word_phrase_translation                      2

Full rows for duplicate entries:
    _id  config.overwrite config.db_collection config.dataset_name  \
42   43                43      llama-typo-eval                 P17   
43   44                44      llama-typo-eval                 P17   
59   61                61      llama-typo-eval                 P17   
60   62                62      llama-typo-eval                 P17   

   config.device    config.exp_id config.max_entries  config.max_new_tokens  \
42          cuda  typo-test-10-01               None                     25   
43          cuda  typo-test-10-01               None                     25   
59          cuda  typo-test-10-01               None                     25   
60